# rlmflow coding-agent walkthrough

Build a recursive coding agent, run a task, save the run, and embed the generated artifact inline. This mirrors `examples/coding/agent.py` but as a notebook so you can poke at every step.

Sibling notebooks read offline against a saved run directory under `examples/_runs/word-search/baseline/`:

- [`node_basics.ipynb`](./node_basics.ipynb) — querying a run with `walk`, `find`, `transcript`, and `persistence.load`
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing a run: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end requires `OPENAI_API_KEY` and live LLM calls. Skip ahead to the other notebooks if you just want to consume a saved run.

## 1. Build the agent

The whole run is one `Flow` driving one root `Node`:

- `Flow` — wires an LLM client (plus optional cheaper alternates registered as `fast`) to a stateful REPL. `start(query)` seeds the run; `async for event in flow.run_streaming(root)` then drives the whole tree, mutating that root in place and yielding one `Event` per tick. (`flow.run(query)` / `await flow.arun(query)` wrap this when you only want the final result.)
- `Runtime` — *where* code runs and *which* tools it has. A `LocalRuntime` runs in-process; `runtime.register_tools(FILE_TOOLS)` exposes the filesystem tools (`read_file`, `write_file`, `edit_file`, `ls`, `grep`, …) to every agent and child — no `Flow` subclass.
- `working_directory` — set it on the runtime and agent code (and the file tools) run inside it; no manual `chdir`. Swap `LocalRuntime` for `DockerRuntime(image, working_directory=...)` to sandbox each step with the same interface.

In [27]:
from pathlib import Path
import shutil
from rlmflow.clients import OpenAIClient
from rlmflow import FILE_TOOLS, Flow, LocalRuntime, persistence, start


def build_agent(
    workdir: Path | str,
    max_depth: int = 2,
    max_iters: int = 30,
    workers: int = 8,
) -> Flow:
    """Construct a coding agent identical t|o examples/coding/agent.py."""
    # The runtime owns where code runs and which tools it has. LocalRuntime runs
    # in-process with the cwd switched into workdir; register_tools exposes the
    # file tools to every agent and child. Swap in DockerRuntime to sandbox.
    runtime = LocalRuntime(working_directory=workdir)
    runtime.register_tools(FILE_TOOLS)
    return Flow(
        OpenAIClient("gpt-5"),
        llm_clients={"fast": OpenAIClient("gpt-5-mini")},
        runtime=runtime,
        max_depth=max_depth,
        max_iters=max_iters,
        workers=workers,
    )

## 2. Run a task

Calling `start(query)` seeds the live trajectory with its root `UserQuery`. `agent.run_streaming(graph)` then drives the whole tree, mutating that same root `Node` in place and yielding one `Event` per tick — one LLM call, one REPL execution, or a wait-resolution. Use `graph.tail()` to inspect the latest node, `graph.finished()` to check for completion, and `graph.agent_result()` for the terminal answer. `replay(graph)` reconstructs per-step snapshots afterward for a timeline.

The agent loop and its side effects are decoupled via **consumers**. Each event is fanned out through a `ConsumerGroup` to small `StreamConsumer` objects — here a notebook renderer that keeps one IPython display handle and `.update`s `render_tree(graph)` each tick (so the tree animates live mid-cell), plus a `GraphCheckpointer` that persists the run to disk on every tick (so a crash still leaves the latest graph saved). The Node tree stays the durable source of truth.


In [28]:
TASK = """Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaScript. They should be fast moving, colorful, and fun.

Delegate subagents for the three artifacts (index.html, style.css, boids.js)
so each file is owned by its own child; as root, wire them together and verify.
No build tools, no libraries, no ES modules.

Requirements:
- index.html — canvas + <script src="..."> tags (no inline JS/CSS)
- style.css — dark background; canvas fills the viewport
- boids.js — hundreds of colorful boids on a 2D canvas; no config UI
- Verify files exist and script tags are ordered correctly before done(...)
"""

WORKDIR = Path("../_runs/notebooks/boids-sim").resolve()
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

agent = build_agent(WORKDIR, max_depth=2)

In [29]:
graph = start(query=TASK)
# Built from the live prompt builder so the preview tracks prompt changes.
prompt = agent.build_system_prompt(graph)
print(prompt)

You are a Recursive Coding Agent: a language model with a user query and important
inputs stored in a Python REPL. You are queried turn-by-turn until you have an
answer. To use the REPL, write code in ```repl``` blocks; it persists across turns.

Available in the REPL:

1. `INPUTS`: a dict of string inputs (may be empty). Your task arrives as the user
   message, not in `INPUTS`. Keys are caller-defined — inspect `list(INPUTS)`
   rather than assuming names; parse JSON with `json.loads(INPUTS["key"])`. Keys
   never shadow REPL variables or tools.
2. `await launch_subagents(specs) -> list`: recursive sub-agent calls. Each spec
   needs a short `query` and may set `inputs` (str -> str), `name`, `model`, and
   `output_schema`. Keep `query` a one/two-sentence instruction and put large
   payloads in `inputs`. Returns a list even for one child.
3. `print(...)`: only stdout is shown back between turns; a bare final expression
   is discarded. Never dump large `INPUTS` values — REPL output 

In [31]:
import asyncio

from IPython.display import display
from rlmflow.consumers import ConsumerGroup, GraphCheckpointer, StreamConsumer
from rlmflow.view import render_tree, replay


# Consumers react to each streamed event; `run_streaming` mutates `graph` in place.
# Jupyter can't use the CLI's ANSI-clear LiveTreeRenderer — instead keep one
# display handle and `.update(...)` it each tick so the tree actually animates
# live (clear_output + print only shows the final frame after the cell ends).
class NotebookTree(StreamConsumer):
    def __init__(self) -> None:
        self._handle = None

    def handle(self, event, graph):
        if graph is None:
            return
        text = render_tree(graph)
        if self._handle is None:
            self._handle = display(text, display_id=True)
        else:
            self._handle.update(text)


consumers = ConsumerGroup(
    [
        NotebookTree(),
        GraphCheckpointer(WORKDIR / "graph"),  # checkpoint the graph every tick
    ]
)

try:
    async for event in agent.run_streaming(graph):
        consumers.handle(event, graph)
        await asyncio.sleep(0)  # yield so the frontend can paint mid-run
finally:
    consumers.close()  # final flush of the checkpoint

graphs = replay(graph)  # per-step snapshots for the summary + viewer below

In [32]:
# Actual rlm run

In [22]:
current = graph.tail()
print(f"{len(graphs)} snapshots  \u00b7  final: {graph.agent_id} [{current.type}]")
print(f"query : {graph.latest_query().content[:120]!r}...")
print(f"result: {graph.agent_result()[:200]}")

45 snapshots  ·  final: root [done_output]
query : 'Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaScript. They should be fast moving, colorfu'...
result: Created files but verification failed. See console for details on what to fix: boids.js: Must get canvas by id 'boids'.; boids.js: Must use 2D canvas context.; boids.js: Missing requestAnimationFrame 


In [34]:
print(render_tree(graph))

Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaSc...
└── root: done Boids simulation is ready. Open index.html in a browser to run ... (2 turns)
    ├── root.index.html author: done {"content": "<!doctype html>\n<html lang=\"en\">\n<head>\n <met... (1 turns)
    ├── root.style.css author: done {"content": "html,body{height:100%;width:100%;margin:0;padding:... (1 turns)
    └── root.boids.js author: done {"content": "(function () {\n 'use strict';\n\n // Fast, colorf... (1 turns)


In [35]:
# The GraphCheckpointer already saved during the run; this is the same call it
# makes — a run directory (manifest + nested per-agent logs) alongside the files
# the agent wrote into WORKDIR. Reload any run dir with `persistence.load(path)`.
run_dir = persistence.save(graph, WORKDIR / "graph", metadata={"task": TASK})
print(f"run -> {run_dir}  ({len(graph.agent_ids())} agents)")

run -> /Users/shyam/Code/rlmkit/examples/_runs/notebooks/boids-sim/graph  (4 agents)


## 3. Preview the generated artifact

Serve the generated `index.html` over local HTTP and embed that URL in the notebook. This exercises the same browser module-loading rules as opening the artifact normally, so import/export mistakes are not hidden by `srcdoc` inlining.

In [36]:
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import IFrame
import socket
import threading

html_path = (WORKDIR / "index.html").resolve()
if not html_path.is_file():
    candidates = sorted(p for p in WORKDIR.glob("**/index.html") if "graph" not in p.parts)
    if not candidates:
        raise FileNotFoundError(f"no index.html under {WORKDIR}")
    html_path = candidates[0].resolve()
site_root = html_path.parent

_prev = globals().get("preview_server")
if _prev is not None:
    _prev.shutdown()
    _prev.server_close()

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]


class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass


handler = partial(QuietHandler, directory=str(site_root))
preview_server = ThreadingHTTPServer(("127.0.0.1", port), handler)
threading.Thread(target=preview_server.serve_forever, daemon=True).start()

url = f"http://127.0.0.1:{port}/{html_path.name}"
print(f"serving {site_root} -> {url}")
display(IFrame(url, width="100%", height=600))

serving /Users/shyam/Code/rlmkit/examples/_runs/notebooks/boids-sim -> http://127.0.0.1:63358/index.html


## 4. Open the interactive viewer

`open_viewer(source)` boots a Gradio stepper (step slider + clickable graph + per-agent transcript) inline.

In [37]:
from rlmflow.view import open_viewer

open_viewer(WORKDIR / "graph", inline=True, quiet=True)

## 5. Render frames of each step

In [ ]:
# save_steps(...) below is the supported path: it writes one PNG per step into
# WORKDIR/frames (handy for GIFs / blog strips). save_image(...) writes a single frame.

## Next

- [`node_basics.ipynb`](./node_basics.ipynb) — walk the saved trace with the `Node` query API.
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — inline Plotly, Mermaid, DOT, Gantt, report exports.